# Notebook 1: Terraform Provisioning (Pre-`node1`)

Run this notebook **before** `node1` is accessible. It provisions Chameleon infrastructure from `devops/tf/kvm` and prepares the `node1` entrypoint used by the cluster bootstrap notebook.

Manual supervision points:
- Validate Terraform plan output before apply.
- Confirm Terraform outputs contain the expected `node1` access details.

## Chameleon Credentials Requirement

Before provisioning, obtain Chameleon Cloud application credentials and place them in:

- `~/.config/openstack/clouds.yaml`

Terraform provisioning in this notebook **cannot proceed** without that file.

In [ ]:
set -euo pipefail

REPO_ROOT="${REPO_ROOT:-$HOME/recipe-scraper-mlops}"
TF_DIR="${TF_DIR:-$REPO_ROOT/devops/tf/kvm}"
ANSIBLE_DIR="${ANSIBLE_DIR:-$REPO_ROOT/devops/ansible}"
KUBESPRAY_DIR="${KUBESPRAY_DIR:-$ANSIBLE_DIR/k8s/kubespray}"
KUBESPRAY_INVENTORY="${KUBESPRAY_INVENTORY:-$ANSIBLE_DIR/k8s/inventory/mycluster/hosts.yaml}"

TF_OPENSTACK_CLOUD="${TF_OPENSTACK_CLOUD:-kvm_tacc}"
TF_OPENSTACK_REGION="${TF_OPENSTACK_REGION:-KVM@TACC}"
TF_OPENSTACK_ENDPOINT_TYPE="${TF_OPENSTACK_ENDPOINT_TYPE:-public}"
TF_SUFFIX="${TF_SUFFIX:-proj22}"
TF_CPU_FLAVOR_ID="${TF_CPU_FLAVOR_ID:-7df7c35e-b47d-4164-a9b7-148ba76885f3}"
TF_CREATE_GPU_NODE="${TF_CREATE_GPU_NODE:-false}"

cat <<'EOF'
Parameter summary:
  REPO_ROOT            = $REPO_ROOT
  TF_DIR               = $TF_DIR
  ANSIBLE_DIR          = $ANSIBLE_DIR
  KUBESPRAY_DIR        = $KUBESPRAY_DIR
  KUBESPRAY_INVENTORY  = $KUBESPRAY_INVENTORY
  TF_OPENSTACK_CLOUD   = $TF_OPENSTACK_CLOUD
  TF_OPENSTACK_REGION  = $TF_OPENSTACK_REGION
  TF_ENDPOINT_TYPE     = $TF_OPENSTACK_ENDPOINT_TYPE
  TF_SUFFIX            = $TF_SUFFIX
  TF_CPU_FLAVOR_ID     = $TF_CPU_FLAVOR_ID
  TF_CREATE_GPU_NODE   = $TF_CREATE_GPU_NODE
EOF

In [ ]:
set -euo pipefail

for cmd in terraform bash; do
  command -v "$cmd" >/dev/null || {
    echo "Missing required command: $cmd" >&2
    exit 1
  }
done

CLOUDS_YAML="${HOME}/.config/openstack/clouds.yaml"
if [[ ! -f "$CLOUDS_YAML" ]]; then
  echo "Missing credentials file: $CLOUDS_YAML" >&2
  exit 1
fi

for path in \
  "$TF_DIR/main.tf" \
  "$TF_DIR/variables.tf" \
  "$TF_DIR/outputs.tf" \
  "$TF_DIR/provider.tf"; do
  [[ -f "$path" ]] || {
    echo "Missing expected Terraform file: $path" >&2
    exit 1
  }
done

echo "Prerequisites passed."

## Terraform Phase

Cells below run `terraform init`, `plan`, and `apply` in `devops/tf/kvm` using the parameter values from this notebook.

In [ ]:
set -euo pipefail

cd "$TF_DIR"
terraform init

In [ ]:
set -euo pipefail

cd "$TF_DIR"
terraform plan \
  -var="openstack_cloud=${TF_OPENSTACK_CLOUD}" \
  -var="openstack_region=${TF_OPENSTACK_REGION}" \
  -var="openstack_endpoint_type=${TF_OPENSTACK_ENDPOINT_TYPE}" \
  -var="suffix=${TF_SUFFIX}" \
  -var="cpu_flavor_id=${TF_CPU_FLAVOR_ID}" \
  -var="create_gpu_node=${TF_CREATE_GPU_NODE}"

In [ ]:
set -euo pipefail

cd "$TF_DIR"
terraform apply \
  -var="openstack_cloud=${TF_OPENSTACK_CLOUD}" \
  -var="openstack_region=${TF_OPENSTACK_REGION}" \
  -var="openstack_endpoint_type=${TF_OPENSTACK_ENDPOINT_TYPE}" \
  -var="suffix=${TF_SUFFIX}" \
  -var="cpu_flavor_id=${TF_CPU_FLAVOR_ID}" \
  -var="create_gpu_node=${TF_CREATE_GPU_NODE}"

In [ ]:
set -euo pipefail

cd "$TF_DIR"
terraform output || true

## Validation + Recovery

Validation checklist:
- Terraform apply completed successfully.
- Terraform outputs include values needed to reach `node1`.

Recovery note:
- If apply partially fails, use the cleanup guidance in `devops/tf/kvm/README.md` before retrying.

## Handoff to `node1`

Next steps:
1. SSH into `node1` (the control host/public entrypoint).
2. Clone this repository onto `node1`.
3. `cd` into repository root on `node1`.
4. Continue with Notebook 2: `02_node1_cluster_bootstrap.ipynb`.